**Cấu hình**

In [ ]:
import os
# --- CELL 1: CẤU HÌNH KẾT NỐI ---
STORAGE_ACCOUNT_NAME = "nhatlong" 
# DÁN KEY CỦA BẠN VÀO ĐÂY:
STORAGE_ACCOUNT_KEY  = os.environ["AZURE_STORAGE_KEY"]

spark.conf.set(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", STORAGE_ACCOUNT_KEY)

# Tên các bảng cần tối ưu
TBL_SILVER = "silver_server_telemetry"
TBL_GOLD   = "gold_datacenter_stats"

print("✅ Sẵn sàng cho thực nghiệm Tối ưu hóa.")

**THỰC NGHIỆM 1 - TỐI ƯU HÓA (Z-ORDERING)**

_Đo thời gian "Trước khi tối ưu"_

In [ ]:
%sql
-- --- CELL 2: BENCHMARK TRƯỚC TỐI ƯU ---
-- Hãy chạy lệnh này và NHÌN XUỐNG GÓC DƯỚI CÙNG BÊN PHẢI của kết quả
-- Bạn sẽ thấy dòng: "Command took X.XX seconds" -> GHI LẠI SỐ NÀY.

SELECT * FROM silver_server_telemetry 
WHERE location = 'Ha Noi' 
AND cpu_usage > 80;

-- (Giả sử mất: 2.5 giây)

_Thực hiện Tối ưu hóa (Z-Ordering)_

In [ ]:
# --- CELL 3 (ĐÃ SỬA LỖI): CHẠY TỐI ƯU HÓA & BÁO CÁO KẾT QUẢ ---
from pyspark.sql.functions import col
import time

print(f"⏳ Đang thực hiện Z-Ordering cho bảng {TBL_SILVER}... (Vui lòng đợi 1-2 phút)")
start_time = time.time()

# 1. Chạy lệnh Optimize
df_opt_results = spark.sql(f"OPTIMIZE {TBL_SILVER} ZORDER BY (location)")

# 2. Lấy dữ liệu từ cột 'metrics' (Sửa lỗi tại đây)
# Dữ liệu trả về có dạng: Row(path=..., metrics=Row(numFilesAdded=1, numFilesRemoved=5...))
row = df_opt_results.select("metrics").collect()[0]
metrics = row["metrics"]

# 3. Trích xuất các chỉ số từ bên trong 'metrics'
# Lưu ý: Nếu metrics là None (trường hợp hiếm), ta gán bằng 0
if metrics:
    files_added = metrics["numFilesAdded"]
    files_removed = metrics["numFilesRemoved"]
    deletion_vector = metrics["numDeletionVectorsRemoved"] if "numDeletionVectorsRemoved" in metrics else 0
else:
    files_added = 0
    files_removed = 0

end_time = time.time()
duration = round(end_time - start_time, 2)

# 4. IN BÁO CÁO ĐẸP (REPORT CARD)
print("\n" + "="*55)
print(f"   📊 BÁO CÁO TỐI ƯU HÓA (OPTIMIZATION REPORT)")
print("="*55)
print(f"🎯 Bảng mục tiêu    : {TBL_SILVER}")
print(f"⏱️ Thời gian chạy   : {duration} giây")
print("-" * 55)

if files_added == 0 and files_removed == 0:
    print("⚠️  Bảng đã được tối ưu sẵn. Không có gì thay đổi.")
else:
    print(f"✅ Số file MỚI tạo ra (Gọn gàng) : {files_added}")
    print(f"🗑️ Số file CŨ bị xóa (Rác/Nhỏ)   : {files_removed}")
    
    if files_removed > 0:
        ratio = round(files_removed / files_added, 1) if files_added > 0 else files_removed
        print(f"⚡ Tỷ lệ nén (Compaction Ratio) : 1 file mới thay thế cho {ratio} file nhỏ cũ.")

print("-" * 55)
print("💡 Ý NGHĨA KỸ THUẬT:")
print("   Z-Ordering đã sắp xếp lại dữ liệu vật lý theo 'location'.")
print("   Spark sẽ gom các file nhỏ rải rác thành file lớn và sắp xếp")
print("   để các bản ghi cùng 'Hà Nội' nằm cạnh nhau trên ổ đĩa.")
print("="*55)

_Đo thời gian "Sau khi tối ưu"_

In [ ]:
%sql
-- --- CELL 4: BENCHMARK SAU TỐI ƯU ---
-- Chạy lại đúng câu lệnh ở Cell 2

SELECT * FROM silver_server_telemetry 
WHERE location = 'Ha Noi' 
AND cpu_usage > 80;

-- So sánh thời gian: Thường sẽ nhanh hơn (Ví dụ: còn 0.5 giây).
-- Lưu ý: Với lượng dữ liệu nhỏ của đồ án, sự chênh lệch có thể chỉ là mili-giây, 
-- nhưng quan trọng là quy trình thực hiện đúng.

**THỰC NGHIỆM 2 - QUẢN TRỊ LƯU TRỮ (VACUUM)**

_Dọn dẹp file rác_

In [ ]:
# --- CELL 5 (MỚI): DỌN DẸP LƯU TRỮ & BÁO CÁO DUNG LƯỢNG ---

# 1. Tắt cơ chế bảo vệ (Bắt buộc để dọn dẹp ngay lập tức)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

print("🔍 Đang quét hệ thống để tìm file rác (Unreferenced files)...")

# 2. Chạy thử (DRY RUN) để đếm số lượng file rác sẽ bị xóa
# DRY RUN trả về danh sách các file rác nhưng CHƯA xóa
df_dry_run = spark.sql(f"VACUUM {TBL_SILVER} RETAIN 0 HOURS DRY RUN")
trash_count = df_dry_run.count()

print(f"⚠️  Phát hiện {trash_count} file thừa (rác) không còn sử dụng.")

# 3. Chạy thật (EXECUTE)
if trash_count > 0:
    print("🧹 Đang tiến hành dọn dẹp (Deleting)...")
    spark.sql(f"VACUUM {TBL_SILVER} RETAIN 0 HOURS")
    
    print("\n" + "="*50)
    print(f"   🧹 BÁO CÁO QUẢN TRỊ LƯU TRỮ (STORAGE GOVERNANCE)")
    print("="*50)
    print(f"✅ Trạng thái       : HOÀN THÀNH (SUCCESS)")
    print(f"🗑️ Đã xóa vĩnh viễn : {trash_count} file rác")
    print("-" * 50)
    print("💡 Ý NGHĨA KỸ THUẬT:")
    print("   Các file này là phiên bản cũ do quá trình Streaming/Optimize tạo ra.")
    print("   Việc xóa chúng giúp giải phóng dung lượng Storage và giảm chi phí Cloud.")
    print("   Lưu ý: Sau khi xóa, không thể dùng Time Travel về thời điểm cũ hơn 0 giờ.")
    print("="*50)
else:
    print("✅ Hệ thống sạch sẽ! Không có file rác nào để dọn.")

**THỰC NGHIỆM 3 - DU HÀNH THỜI GIAN (TIME TRAVEL)**

_Kịch bản demo: "Sếp lỡ tay xóa nhầm dữ liệu quan trọng. Hệ thống khôi phục lại như thế nào?"_

_Mô phỏng sự cố và Khôi phục_

In [ ]:
%sql
-- --- CELL 6: DEMO TIME TRAVEL ---

-- 1. Xem lịch sử các phiên bản của bảng
DESCRIBE HISTORY silver_server_telemetry;
-- (Bạn sẽ thấy các version: 0, 1, 2... Cột 'operation' ghi là STREAMING UPDATE, OPTIMIZE...)

-- 2. Giả sử ta muốn xem dữ liệu của phiên bản gốc ban đầu (Version 0)
SELECT * FROM silver_server_telemetry VERSION AS OF 0 LIMIT 10;

-- 3. Hoặc xem dữ liệu cách đây 10 phút (khi chưa bị lỗi)
-- SELECT * FROM silver_server_telemetry TIMESTAMP AS OF '2023-11-25T10:00:00.000Z';

**_CLONING (Sao chép bảng để Test)_**

_Kịch bản: Nhóm phát triển muốn thử nghiệm code mới nhưng sợ làm hỏng bảng dữ liệu thật (silver_server_telemetry). Giải pháp là tạo bản sao (CLONE)._

Deep Clone (Sao chép sâu)

- Tạo bảng sao (DEEP CLONE).
- So sánh số lượng bản ghi (Count Check).
- So sánh mẫu dữ liệu (Data Sample Check).
- Thử xóa dữ liệu ở bảng Clone để chứng minh bảng Gốc không bị ảnh hưởng.

In [ ]:
# --- CELL 7: TẠO BẢN SAO KIỂM THỬ & SO SÁNH (CLONING & VERIFICATION) ---
from pyspark.sql.functions import col

# Tên bảng
tbl_original = TBL_SILVER
tbl_clone    = f"{TBL_SILVER}_TEST_CLONE"

print(f"🚀 Đang tạo bản sao DEEP CLONE từ bảng '{tbl_original}' sang '{tbl_clone}'...")

# 1. Thực hiện CLONE
# DEEP CLONE: Sao chép cả dữ liệu lẫn metadata (Độc lập hoàn toàn)
spark.sql(f"CREATE OR REPLACE TABLE {tbl_clone} DEEP CLONE {tbl_original}")

# 2. KIỂM TRA & SO SÁNH (VERIFICATION)
print("\n" + "="*60)
print("   🔍 BÁO CÁO SO SÁNH (ORIGINAL vs CLONE)")
print("="*60)

# A. So sánh số lượng dòng (Row Count)
count_original = spark.read.table(tbl_original).count()
count_clone    = spark.read.table(tbl_clone).count()

print(f"1️⃣ SO SÁNH SỐ LƯỢNG (ROW COUNT):")
print(f"   - Bảng Gốc ({tbl_original}) : {count_original} dòng")
print(f"   - Bảng Clone ({tbl_clone})  : {count_clone} dòng")

if count_original == count_clone:
    print("   ✅ KẾT QUẢ: KHỚP 100% (MATCHED)")
else:
    print("   ❌ KẾT QUẢ: KHÔNG KHỚP (MISMATCHED)")

print("-" * 60)

# B. Thử nghiệm tính độc lập (Isolation Test)
print(f"2️⃣ THỬ NGHIỆM TÍNH ĐỘC LẬP (ISOLATION TEST):")
print("   👉 Kịch bản: Xóa thử 1 dòng bên bảng CLONE xem bảng GỐC có bị mất không?")

# Xóa 1 dòng bất kỳ bên Clone
spark.sql(f"DELETE FROM {tbl_clone} WHERE server_id = (SELECT server_id FROM {tbl_clone} LIMIT 1)")

# Đếm lại
count_original_after = spark.read.table(tbl_original).count()
count_clone_after    = spark.read.table(tbl_clone).count()

print(f"   - Sau khi xóa bên Clone:")
print(f"     + Clone còn : {count_clone_after} dòng (Đã giảm 1)")
print(f"     + Gốc còn   : {count_original_after} dòng (Vẫn nguyên vẹn)")

if count_original_after > count_clone_after:
    print("   ✅ KẾT QUẢ: THÀNH CÔNG! Bảng Gốc an toàn tuyệt đối.")
else:
    print("   ❌ KẾT QUẢ: THẤT BẠI!")

print("="*60)
print("💡 Ý NGHĨA KỸ THUẬT:")
print("   Tính năng 'DEEP CLONE' cho phép tạo môi trường Staging/Testing ngay lập tức")
print("   mà không cần chạy lại quy trình ETL. Rất hữu ích cho CI/CD và Debug lỗi.")
print("="*60)

**AUDIT LOG (Kiểm tra lịch sử thao tác)**

Kiểm tra lịch sử thao tác

In [ ]:
%sql
-- --- CELL 8: AUDIT LOG ---
SELECT 
    version,
    timestamp,
    userId, -- Ai làm?
    operation, -- Làm gì? (WRITE, OPTIMIZE, VACUUM)
    operationParameters
FROM (DESCRIBE HISTORY silver_server_telemetry)
ORDER BY version DESC;